# Lesson 4.2 — CloudFormation Advanced: StackSets, Nested Stacks & CDK

**Level: Intermediate** | **Exam:** SOA-C03 Domain 3, Skills 3.1.2, 3.1.4

## Learning Objectives
- Use nested stacks to organize complex infrastructure
- Deploy CloudFormation StackSets across multiple accounts and regions
- Write AWS CDK applications in Python that synthesize CloudFormation templates
- Use cross-stack references (Outputs + ImportValue) to share resources

_Reference: [CloudFormation StackSets](https://docs.aws.amazon.com/cloudformation/latest/userguide/what-is-cfnstacksets.html) | [AWS CDK](https://docs.aws.amazon.com/cdk/v2/guide/)_

## Nested Stacks

Large templates become hard to manage. **Nested stacks** let you break templates into reusable modules:

```
Root Stack (main.yaml)
├── VPC Stack (vpc.yaml)           → Creates VPC, subnets, route tables
├── Security Stack (security.yaml) → Creates security groups, IAM roles  
├── RDS Stack (database.yaml)      → References VPC + Security outputs
└── App Stack (application.yaml)  → References all above
```

- Parent stack creates child stacks as `AWS::CloudFormation::Stack` resources
- Pass values between stacks via `Parameters` and `Outputs`
- Child stacks must be uploaded to S3 first

## CloudFormation StackSets

StackSets deploy a **single template to multiple accounts and regions** in one operation:

- **Service-managed**: AWS Organizations manages account targeting
- **Self-managed**: You specify target account IDs and regions manually
- Use case: Deploy security baseline, logging config, or guardrails org-wide

```
Stack Set → Deploy to: [dev account, staging account, prod account]
         → In regions: [us-east-1, eu-west-1]
= 6 stack instances created automatically
```

## AWS CDK (Cloud Development Kit)

CDK lets you define infrastructure using **real programming languages** (Python, TypeScript, Java, C#) instead of YAML:

```python
import aws_cdk as cdk
from aws_cdk import aws_s3 as s3

class MyStack(cdk.Stack):
    def __init__(self, scope, id, **kwargs):
        super().__init__(scope, id, **kwargs)
        
        bucket = s3.Bucket(self, 'MyBucket',
            versioned=True,
            encryption=s3.BucketEncryption.S3_MANAGED,
            removal_policy=cdk.RemovalPolicy.DESTROY
        )
```

`cdk synth` → generates CloudFormation template
`cdk deploy` → deploys to AWS
`cdk diff` → shows what will change (like change sets)

In [ ]:
import boto3, json, subprocess

cf = boto3.client('cloudformation', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
%%writefile networking-stack.yaml
AWSTemplateFormatVersion: '2010-09-09'
Description: 'Nested Stack: Networking Layer'

Parameters:
  EnvironmentName:
    Type: String
    Default: dev

Resources:
  # In a real template, you'd create VPC, subnets, etc.
  # For demo, we create a simple security group placeholder
  DemoSG:
    Type: AWS::EC2::SecurityGroup
    Properties:
      GroupName: !Sub '${EnvironmentName}-demo-sg'
      GroupDescription: 'Demo security group from nested stack'
      Tags:
        - Key: Environment
          Value: !Ref EnvironmentName
        - Key: CreatedBy
          Value: 'NestedStack'

Outputs:
  SecurityGroupId:
    Value: !Ref DemoSG
    Export:
      Name: !Sub '${AWS::StackName}-SGID'

In [ ]:
# Cross-Stack References — share resources between independent stacks
# Pattern: Stack A exports a value; Stack B imports it with !ImportValue

print('Cross-Stack Reference Pattern:')
print()
print('Stack A (networking):')
print('  Outputs:')
print('    VpcId:')
print('      Value: !Ref MyVPC')
print('      Export:')
print("        Name: !Sub '${AWS::StackName}-VpcId'")  
print()
print('Stack B (application):')
print('  Resources:')
print('    MyInstance:')
print('      Properties:')
print("        VpcId: !ImportValue 'networking-stack-VpcId'")
print()
print('Key rules:')
print('  - Export names must be UNIQUE per region')
print('  - Cannot delete a stack if another stack imports its exports')
print('  - Use !Sub to create unique export names: ${AWS::StackName}-VpcId')
print()

# List existing exported values in the account
exports = cf.list_exports()['Exports']
print(f'Current CloudFormation Exports ({len(exports)}):')
for exp in exports[:10]:
    print(f'  {exp["ExportingStackId"].split(":")[-1].split("/")[1]:30} → {exp["Name"]}')
if not exports:
    print('  (none — create a stack with Outputs to see exports here)')

In [ ]:
# Install AWS CDK (if not already installed)
result = subprocess.run('cdk --version', shell=True, capture_output=True, text=True)
if result.returncode == 0:
    print(f'CDK version: {result.stdout.strip()}')
else:
    print('CDK not installed. Installing...')
    print('Run: npm install -g aws-cdk')
    print('Then: pip install aws-cdk-lib constructs')
    print()
    print('CDK requires Node.js (for the CLI) and Python (for writing CDK code)')

# Show what a Python CDK app looks like
print('\nExample CDK Python App:')
print('='*50)

In [ ]:
%%writefile cloudops_cdk_example.py
# Example AWS CDK Python App
# Run: pip install aws-cdk-lib constructs
# Then: cdk synth (to generate CloudFormation template)

import aws_cdk as cdk
from aws_cdk import (
    aws_s3 as s3,
    aws_iam as iam,
    aws_cloudwatch as cloudwatch,
)
from constructs import Construct

class CloudOpsLearningStack(cdk.Stack):
    """
    CDK Stack that creates:
    - S3 bucket with encryption, versioning, and lifecycle
    - IAM role for EC2 access
    - CloudWatch alarm on bucket requests
    """
    
    def __init__(self, scope: Construct, construct_id: str, env_name: str, **kwargs) -> None:
        super().__init__(scope, construct_id, **kwargs)
        
        # S3 Bucket
        bucket = s3.Bucket(
            self, 'ArtifactBucket',
            bucket_name=f'cdk-cloudops-{env_name}-artifacts',
            versioned=(env_name == 'prod'),      # Version only in prod
            encryption=s3.BucketEncryption.S3_MANAGED,
            block_public_access=s3.BlockPublicAccess.BLOCK_ALL,
            removal_policy=cdk.RemovalPolicy.DESTROY,
            lifecycle_rules=[
                s3.LifecycleRule(
                    id='ExpireOldVersions',
                    noncurrent_version_expiration=cdk.Duration.days(30)
                )
            ]
        )
        
        # IAM Role
        role = iam.Role(
            self, 'EC2Role',
            assumed_by=iam.ServicePrincipal('ec2.amazonaws.com'),
            managed_policies=[
                iam.ManagedPolicy.from_aws_managed_policy_name('AmazonSSMManagedInstanceCore')
            ]
        )
        
        # Grant bucket access to the role
        bucket.grant_read_write(role)
        
        # CloudFormation Output
        cdk.CfnOutput(self, 'BucketName', value=bucket.bucket_name)
        cdk.CfnOutput(self, 'RoleArn', value=role.role_arn)


app = cdk.App()
CloudOpsLearningStack(app, 'CloudOpsLearning-Dev', env_name='dev')
CloudOpsLearningStack(app, 'CloudOpsLearning-Prod', env_name='prod')
app.synth()

print('CDK example written. Run: python cloudops_cdk_example.py && cdk synth')

## Exercise

In [ ]:
# TODO: Using boto3, list all CloudFormation StackSets in your account
# For each StackSet, show:
# - StackSet name and status
# - Number of stack instances (across all accounts/regions)
# If no StackSets exist, print a sample StackSet creation command

import boto3
cf = boto3.client('cloudformation', region_name='us-east-1')

# YOUR CODE HERE

In [ ]:
# Cleanup
import os
for f in ['networking-stack.yaml', 'cloudops_cdk_example.py']:
    if os.path.exists(f):
        os.remove(f)
        print(f'Removed: {f}')

## Summary
- **Nested stacks** decompose large templates into reusable modules
- **Cross-stack references**: Export from one stack, `!ImportValue` in another
- **StackSets** deploy templates org-wide across accounts and regions
- **CDK**: Write infrastructure in Python/TypeScript; `cdk synth` generates CloudFormation
- CDK is an **abstraction on top of** CloudFormation — it always produces a CloudFormation template

---
**Next:** [Lesson 4.3 — Deployment Strategies: Blue/Green, Rolling, Canary](lesson-4.3-deployment-strategies.ipynb)